# Phase 2 — Chunking Strategies

**Goal:** Turn page-level text into retrievable chunks that (a) fit in an LLM context window, (b) carry enough surrounding context to be self-contained, and (c) preserve `{doc_id, page, section, chunk_id}` for citations.

We compare three strategies:

1. **Fixed-size character chunking** — naive baseline.
2. **Recursive character splitting** — LangChain's default. Splits on paragraph/sentence/word boundaries.
3. **Token-aware splitting** — uses `tiktoken` so chunks fit a known token budget.

We will see why **chunk size 500–800 tokens with ~100-token overlap** is the canonical sweet spot for research papers.


## 2.1 — Load the parsed PDF from Phase 1


In [ ]:
import os, sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "app").is_dir() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

import fitz, hashlib
from pydantic import BaseModel, Field
from typing import Optional

class PageRecord(BaseModel):
    doc_id: str
    page: int
    text: str
    n_images: int = 0
    source: str
    section: Optional[str] = None

class PdfDocument(BaseModel):
    doc_id: str
    source: str
    title: Optional[str] = None
    n_pages: int
    pages: list[PageRecord] = Field(default_factory=list)

def _doc_id(path: Path) -> str:
    return f"{path.stem}_{hashlib.sha1(path.read_bytes()).hexdigest()[:10]}"

def load_pdf(path: str | Path) -> PdfDocument:
    path = Path(path)
    d = fitz.open(path)
    return PdfDocument(
        doc_id=_doc_id(path),
        source=path.name,
        title=d.metadata.get("title") or None,
        n_pages=d.page_count,
        pages=[
            PageRecord(doc_id=_doc_id(path), page=i, text=p.get_text("text"),
                       n_images=len(p.get_images(full=True)), source=path.name)
            for i, p in enumerate(d, start=1)
        ],
    )

pdf = load_pdf("data/raw/attention_is_all_you_need.pdf")
print(f"Loaded {pdf.n_pages} pages from {pdf.source}")


## 2.2 — Define the chunk schema

A chunk is the smallest indexable unit in our system. Every chunk knows where it came from.


In [ ]:
class Chunk(BaseModel):
    chunk_id: str        # e.g. "attention_is_all_you_need_p3_c2"
    doc_id: str
    source: str
    page: int
    text: str
    section: Optional[str] = None
    n_tokens: Optional[int] = None


## 2.3 — Strategy A: Fixed-size character chunking

The naive baseline. Just cut every N characters with M overlap. Cheap, but it splits sentences mid-word and ignores paragraph structure.


In [ ]:
def fixed_chunks(pdf: PdfDocument, size: int = 1500, overlap: int = 200) -> list[Chunk]:
    out = []
    for page in pdf.pages:
        text = page.text
        i = 0
        idx = 0
        while i < len(text):
            piece = text[i:i + size]
            if piece.strip():
                out.append(Chunk(
                    chunk_id=f"{page.doc_id}_p{page.page}_c{idx}",
                    doc_id=page.doc_id,
                    source=page.source,
                    page=page.page,
                    text=piece,
                ))
                idx += 1
            i += size - overlap
    return out

fixed = fixed_chunks(pdf)
print(f"Fixed-size: {len(fixed)} chunks")
print("Example chunk preview:\n")
print(fixed[3].text[:300], "\n...")


## 2.4 — Strategy B: Recursive character splitting

LangChain's `RecursiveCharacterTextSplitter` tries a sequence of separators (`\n\n`, `\n`, `. `, ` `, `""`) and falls back to finer ones only if the chunk is still too large. Result: clean paragraph/sentence boundaries.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def recursive_chunks(pdf: PdfDocument, size: int = 800, overlap: int = 100) -> list[Chunk]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len,
    )
    out = []
    for page in pdf.pages:
        for idx, piece in enumerate(splitter.split_text(page.text)):
            if piece.strip():
                out.append(Chunk(
                    chunk_id=f"{page.doc_id}_p{page.page}_c{idx}",
                    doc_id=page.doc_id,
                    source=page.source,
                    page=page.page,
                    text=piece,
                ))
    return out

recursive = recursive_chunks(pdf)
print(f"Recursive: {len(recursive)} chunks")
print("\nExample:\n", recursive[5].text[:300])


## 2.5 — Strategy C: Token-aware splitting

`tiktoken` lets us count tokens the same way the LLM will. This matters because context windows are measured in tokens, not characters, and the character-to-token ratio varies between models (English prose is ~4 chars/token).


In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")   # used by GPT-4 / 3.5; close enough for Llama too

def token_chunks(pdf: PdfDocument, size: int = 700, overlap: int = 100) -> list[Chunk]:
    out = []
    for page in pdf.pages:
        tokens = enc.encode(page.text)
        i, idx = 0, 0
        while i < len(tokens):
            window = tokens[i:i + size]
            piece = enc.decode(window).strip()
            if piece:
                out.append(Chunk(
                    chunk_id=f"{page.doc_id}_p{page.page}_c{idx}",
                    doc_id=page.doc_id,
                    source=page.source,
                    page=page.page,
                    text=piece,
                    n_tokens=len(window),
                ))
                idx += 1
            i += size - overlap
    return out

tokenized = token_chunks(pdf)
print(f"Token-aware: {len(tokenized)} chunks")
sizes = [c.n_tokens for c in tokenized]
print(f"Token sizes: min={min(sizes)}, median={sorted(sizes)[len(sizes)//2]}, max={max(sizes)}")


## 2.6 — Side-by-side comparison


In [ ]:
import pandas as pd

def summarize(name: str, chunks: list[Chunk]) -> dict:
    char_lens = [len(c.text) for c in chunks]
    return {
        "strategy": name,
        "n_chunks": len(chunks),
        "median_chars": int(pd.Series(char_lens).median()),
        "p95_chars": int(pd.Series(char_lens).quantile(0.95)),
        "first_chunk_starts_clean": chunks[0].text[:1].isupper(),
    }

pd.DataFrame([
    summarize("fixed-1500/200",      fixed),
    summarize("recursive-800/100",   recursive),
    summarize("token-700/100",       tokenized),
])


## 2.7 — Why these defaults?

| Parameter      | Default       | Reasoning                                                                                          |
|----------------|---------------|----------------------------------------------------------------------------------------------------|
| `chunk_size`   | 500–800 tokens| Large enough to contain a complete idea (a paragraph or two), small enough to leave room for ~3–5 retrieved chunks + the question + the answer in a 4k–8k context window. |
| `overlap`      | ~15% of size  | Lets information that straddles a chunk boundary appear in both, so retrieval is more robust.       |
| Separator order| `\n\n` → `\n` → `. ` → ` ` | Paragraph > line > sentence > word. Aim to break at semantic boundaries first.    |

If you swap to a model with a long context (e.g. 128k), you can go up to 1500–2000 token chunks and retrieve fewer of them. But beware: longer chunks dilute embedding-based retrieval ("lost in the middle" effect).


## 2.8 — Semantic chunking (advanced)

The strategies above are *structural*. A more advanced approach is **semantic chunking**: greedily merge adjacent sentences as long as their embeddings stay similar, then start a new chunk when similarity drops. This produces topic-coherent chunks.

We sketch it here using sentence embeddings; we will not make it the default because it is slower and benefits saturate around 700-token chunks for short papers.


In [ ]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def split_sentences(text: str) -> list[str]:
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def semantic_chunks(pdf: PdfDocument, similarity_threshold: float = 0.55, max_chars: int = 1200) -> list[Chunk]:
    out = []
    for page in pdf.pages:
        sentences = split_sentences(page.text)
        if not sentences:
            continue
        embs = embedder.encode(sentences, normalize_embeddings=True)
        current, current_emb = [sentences[0]], embs[0]
        chunk_idx = 0
        for sent, emb in zip(sentences[1:], embs[1:]):
            sim = float(np.dot(current_emb, emb))
            combined = " ".join(current + [sent])
            if sim >= similarity_threshold and len(combined) <= max_chars:
                current.append(sent)
                current_emb = (current_emb + emb) / 2
            else:
                out.append(Chunk(
                    chunk_id=f"{page.doc_id}_p{page.page}_c{chunk_idx}",
                    doc_id=page.doc_id, source=page.source, page=page.page,
                    text=" ".join(current),
                ))
                chunk_idx += 1
                current, current_emb = [sent], emb
        if current:
            out.append(Chunk(
                chunk_id=f"{page.doc_id}_p{page.page}_c{chunk_idx}",
                doc_id=page.doc_id, source=page.source, page=page.page,
                text=" ".join(current),
            ))
    return out

# This is slower; run only the first 5 pages for the demo.
demo_pdf = PdfDocument(**pdf.model_dump())
demo_pdf.pages = pdf.pages[:5]
semantic = semantic_chunks(demo_pdf)
print(f"Semantic: {len(semantic)} chunks over first 5 pages")
print("\nExample:\n", semantic[2].text[:300])


## 2.9 — Save the canonical chunks for downstream notebooks

We will use the **token-aware** chunks (`token_chunks`, size=700, overlap=100) as the default for Phase 3 onwards. Saving them to `data/processed/` so we do not have to recompute.


In [ ]:
import json
processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

chunks_path = processed_dir / "chunks.jsonl"
with chunks_path.open("w", encoding="utf-8") as f:
    for c in tokenized:
        f.write(c.model_dump_json() + "\n")

print(f"Saved {len(tokenized)} chunks to {chunks_path}")
print(f"File size: {chunks_path.stat().st_size/1024:.1f} KB")


## What you learned

- Three chunking strategies and their tradeoffs.
- Why token-aware splitting is the production default.
- The `Chunk` schema with `chunk_id`, `page`, `doc_id` for citations.
- A semantic chunking sketch for advanced use.
- How to persist chunks to JSONL.

## Exercises

1. Implement section-aware chunking: parse Markdown-like headings (`Abstract`, `Introduction`, `Methods`) and never let a chunk straddle a section boundary.
2. Plot a histogram of chunk character lengths for each strategy.
3. Tune `chunk_size` to 400, 800, 1200 — which is best? You will revisit this in Phase 9 with real eval metrics.

## Next: [Phase 3 — Embeddings & FAISS](./2026-05-25-phase03-embeddings-and-faiss.ipynb)
